# FG2027 — first batch: 22 experiments, seed 379
Use a **Google Colab GPU runtime**. Save a copy of this notebook to your Drive.
Run cells in order; **do not use Run all** before reviewing the data audit and pilot.
Order: setup → download → audit → smoke tests → pilot → remaining runs → results.
No numerical tests or experiments have been completed yet.

## 1. Setup
Select **Runtime → Change runtime type → GPU**, then run this cell and approve Drive access.
On a fresh runtime the repository is cloned. An existing clean checkout is updated to FG2027.
Record the printed commit. Do not update code midway through an experiment batch.

In [ ]:
from google.colab import drive, auth
from pathlib import Path
import subprocess, json, sys, os
assert 'COLAB_RELEASE_TAG' in os.environ, 'Run in Google Colab'
assert subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0, 'Select a GPU runtime'
drive.mount('/content/drive')
REPO = Path('/content/fg2027')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch','FG2027',
        'https://github.com/georgebaher/Sign-Language-Recognition-using-Lightweight-Transformers.git',str(REPO)],check=True)
os.chdir(REPO)
assert subprocess.check_output(['git','branch','--show-current'],text=True).strip() == 'FG2027'
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(), 'Checkout has local edits; inspect before updating'
subprocess.run(['git','pull','--ff-only','origin','FG2027'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-fg2027.txt'],check=True)
print('Code commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
subprocess.run(['nvidia-smi'],check=True)

## 2. Find and download the eight data files
Approve Google authentication. Exact paths below select ViTPose landmarks for both datasets.
Only WLASL metadata comes from the MediaPipe folder. Source files remain unchanged.
Downloads go to temporary Colab storage; matching cached files are reused.

In [ ]:
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import hashlib
service = build('drive','v3')
FOLDER_ID = '1tUoqJawvh0-hSxrIrfU3NUFPcQ8Xhusr'
folder_cache = {}
def children(folder):
    if folder not in folder_cache:
        files, page = [], None
        while True:
            result = service.files().list(q=f"'{folder}' in parents and trashed=false",pageToken=page,
                fields='nextPageToken,files(id,name,mimeType,size,md5Checksum)',pageSize=1000,
                supportsAllDrives=True,includeItemsFromAllDrives=True).execute()
            files.extend(result.get('files',[]))
            page = result.get('nextPageToken')
            if not page: break
        folder_cache[folder] = files
    return folder_cache[folder]
def resolve(relative_path):
    parent = FOLDER_ID
    for name in relative_path.split('/'):
        matches = [f for f in children(parent) if f['name']==name]
        assert len(matches)==1, f'Expected exactly one match: {relative_path}; found {len(matches)} for {name}'
        item = matches[0]
        parent = item['id']
    return item
def md5(path):
    digest = hashlib.md5()
    with path.open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''): digest.update(block)
    return digest.hexdigest()
DATA_ROOT = Path('/content/fg2027_data')
source_files = {}
for dataset, prefix, metadata_path in [
    ('wlasl','WLASL_feature_processing_vitpose','WLASL_feature_processing_mediapipe/WLASL_v0.3.json'),
    ('avasag','AVASAG_feature_processing_vitpose','AVASAG_feature_processing_vitpose/avasag_metadata.json')]:
    paths = {'metadata.json':metadata_path}
    paths.update({f'vitpose/{m}_landmarks.parquet':f'{prefix}/data_parquet/{m}_landmarks.parquet'
                  for m in ('hand','pose','face')})
    for local_name, source_path in paths.items():
        meta = resolve(source_path)
        source_files[f'{dataset}/{local_name}'] = {'source_path':source_path,**meta}
        dest = DATA_ROOT / dataset / local_name
        dest.parent.mkdir(parents=True,exist_ok=True)
        if dest.exists() and meta.get('md5Checksum') and md5(dest)==meta['md5Checksum']:
            print('Cached:',source_path)
            continue
        print('Downloading:',source_path,meta.get('size',''),flush=True)
        temporary = dest.with_suffix(dest.suffix+'.tmp')
        with temporary.open('wb') as stream:
            download = MediaIoBaseDownload(stream,service.files().get_media(fileId=meta['id'],supportsAllDrives=True))
            done = False
            while not done: _,done = download.next_chunk()
        if 'size' in meta: assert temporary.stat().st_size==int(meta['size'])
        if 'md5Checksum' in meta: assert md5(temporary)==meta['md5Checksum']
        temporary.replace(dest)
print('All eight files ready.')

## 3. Save/check the one-seed configuration
This uses a separate output directory so an older three-seed configuration cannot be reused silently.
Do not change configuration after training starts. The review flag is set only after the data audit.

In [ ]:
config = json.loads(Path('configs/fg2027.json').read_text())
assert config['seeds']==[379], 'This notebook is for the first, single-seed batch'
CONFIG = Path('/content/drive/MyDrive/FG2027/config_seed379.json')
CONFIG.parent.mkdir(parents=True,exist_ok=True)
if CONFIG.exists():
    saved = json.loads(CONFIG.read_text())
    assert {k:v for k,v in saved.items() if k!='protocol_reviewed'} == {k:v for k,v in config.items() if k!='protocol_reviewed'}, 'Saved config differs; inspect it rather than overwriting'
    config = saved
else:
    CONFIG.write_text(json.dumps(config,indent=2))
OUT = Path(config['output_dir'])
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'source_files.json').write_text(json.dumps(source_files,indent=2))
print(json.dumps(config,indent=2))
from src.fg2027 import matrix
assert len(list(matrix(config)))==22
print('22 planned experiments. Outputs:',OUT)

## 4. Audit — stop here for the first review
Checks feature columns, video/frame alignment, metadata labels, duplicate IDs, missing clips and coordinate ranges.
Reference split counts are WLASL **1442 / 338 / 258**, AVASAG **2596 / 849 / 890** (train / validation / test).
Expected feature widths are hands **84**, pose **26**, face **136**. Exact pose-point selection and extraction provenance still need verification.
Send the printed output or error for review before continuing. Never filter clips or change splits simply to make the audit pass.

In [ ]:
def execute(action,*options):
    subprocess.run([sys.executable,'-u','-m','src.fg2027',action,'--config',str(CONFIG),*options],check=True)
execute('audit')
expected = {'wlasl':{'train':1442,'val':338,'test':258},'avasag':{'train':2596,'val':849,'test':890}}
for dataset in config['datasets']:
    report = json.loads((OUT/dataset/'data_audit.json').read_text())
    print(dataset,json.dumps({k:report[k] for k in ('counts','missingness','length_min','length_max')},indent=2))
    print('Feature widths:',{k:len(v) for k,v in report['columns'].items()})
    assert report['counts']==expected[dataset], 'Split counts differ from paper: investigate before training'

## 5. Smoke tests
After reviewing the audit, set `DATA_REVIEWED=True`. These checks run small synthetic cases and short real train/validation batches for all 11 configurations on each dataset.
They do not produce research results or establish that full-length batches will fit in GPU memory.
Continue only if both datasets finish without errors and every configuration reports SMOKE PASS.

In [ ]:
DATA_REVIEWED = False
assert DATA_REVIEWED, 'Review both data audits and confirm the pose indices first'
config['protocol_reviewed'] = True
CONFIG.write_text(json.dumps(config,indent=2))
execute('smoke')

## 6. Pilot — one complete experiment
Run WLASL + encoder-only + hands + seed 379 for the full 100 epochs. This is one of the 22 experiments, not an extra run.
Review training/validation curves, runtime and checkpoint size before the batch. Test scores must not guide hyperparameter changes.
If training stops, rerun this cell with `--resume`; only the unfinished epoch is repeated.

In [ ]:
execute('run','--dataset','wlasl','--model','encoder','--modalities','H','--seed','379','--resume')
pilot = OUT/'wlasl/encoder_H/seed_379'
history = json.loads((pilot/'history.json').read_text())
import matplotlib.pyplot as plt
fig,axes = plt.subplots(1,2,figsize=(10,3))
for split in ('train','val'):
    axes[0].plot([r['epoch'] for r in history],[r[f'{split}_loss'] for r in history],label=split)
    axes[1].plot([r['epoch'] for r in history],[r[f'{split}_acc'] for r in history],label=split)
for ax,title in zip(axes,['Loss','Accuracy']):
    ax.set(xlabel='Epoch',title=title); ax.legend()
plt.tight_layout(); plt.show()
metrics = json.loads((pilot/'metrics.json').read_text())
print({k:metrics[k] for k in ('best_epoch','training_seconds','peak_cuda_memory_bytes','parameters')})
print('Checkpoint MB:',{name:round((pilot/name).stat().st_size/1e6,1) for name in ('best.pt','last.pt')})

## 7. Remaining experiments — run one dataset at a time
Set `PILOT_REVIEWED=True` after reviewing the pilot. WLASL has 10 remaining runs because the completed pilot is skipped. AVASAG has 11 runs.
Run sequentially in one GPU runtime. Do not run both cells concurrently in separate sessions sharing the output folder.
If a run fails, stop and inspect the error. Do not silently change batch size, sequence length or the dataset.
After a runtime reset: repeat setup, downloads, config, audit and smoke cells, then rerun the interrupted dataset cell.
Resume requires the same code/config/data and recorded Python/PyTorch/CUDA/GPU type. If that check fails, ask for review.

In [ ]:
PILOT_REVIEWED = False
assert PILOT_REVIEWED, 'Review pilot learning curves, runtime and storage before the batch'
execute('run','--dataset','wlasl','--resume')

In [ ]:
assert PILOT_REVIEWED, 'Review the pilot first'
execute('run','--dataset','avasag','--resume')

## 8. Results
Run this cell after the pilot, after a dataset finishes, or after an interrupted batch to see progress.
With one seed, report scores only; there is no across-seed standard deviation or confidence interval.
Keep per-video predictions for the planned later bootstrap analysis. Additional seeds are deferred.

In [ ]:
execute('summarize')
from IPython.display import Markdown,display
display(Markdown((OUT/'results.md').read_text()))
print('Per-run table:',OUT/'results.csv')